# Reconhecimento de atividades humanas: e quando o modelo encontra pessoas inéditas?

**Projeto P01B — Estatística & Ciência de Dados · USP/ICMC**  
**Pergunta:** separar janelas aleatoriamente superestima o desempenho em relação a deixar pessoas inteiras fora do treinamento?

> Regras científicas: os **10.299 exemplos são janelas**, não 10.299 pessoas. Existem 30 pessoas, e a separação oficial reserva nove delas para teste. O teste não seleciona nenhum algoritmo ou hiperparâmetro. Não inventamos resultados quando o ZIP UCI está indisponível.

**Fonte primária:** [UCI Dataset 240](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones), DOI `10.24432/C54S4K`. Veja `docs/DOSSIER_HAR.md`, `docs/PROTOCOL.md` e `LICENSE_DATA_NOTICE.md` para método, proveniência e conflito de licença.

## 0. Ambiente e configurações

Rode com kernel limpo **a partir da raiz do repositório**. `MODE='full'` usa os grids curtos e 6 famílias; `'quick'` testa as seis famílias com um candidato cada, útil para inspeção de infraestrutura. O notebook não muda de protocolo sozinho em caso de lentidão.

O download requer internet. Em ambientes offline, coloque o ZIP oficial em `data/raw/uci_har_official.zip`. Sem ele, o notebook deve parar antes de apresentar métricas.

In [ ]:
from pathlib import Path
import sys, platform, json
import numpy as np, pandas as pd, sklearn
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    # Permite notebook iniciado de dentro de notebooks/ em JupyterLab.
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Abra o projeto na raiz para reproduzir"
sys.path.insert(0, str(ROOT / "src"))
from har_subject_shift.data import download, audit, load_split, require_audit, LABELS
from har_subject_shift.experiment import train, final_test
from har_subject_shift.visualize import generate
MODE = "full"  # para teste técnico rápido: "quick"
print({"Python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"sklearn":sklearn.__version__,"mode":MODE})

## 1. G0 — Fonte, integridade e licença

O ZIP oficial é baixado em arquivo temporário. O projeto confere CRC, SHA-256 e rejeita caminhos perigosos. **Nenhuma cópia alternativa substitui automaticamente o dataset primário**.

O catálogo UCI mostra CC BY 4.0, mas há restrição histórica no README. A discrepância é registrada e **não é interpretada como autorização de exploração comercial**.

In [ ]:
official_zip = ROOT / "data/raw/uci_har_official.zip"
if not official_zip.exists():
    try:
        download(ROOT)
    except Exception as exc:
        raise RuntimeError("G0 BLOQUEADO: sem ZIP oficial e download indisponível. Transfira-o manualmente para data/raw/uci_har_official.zip. Sem dados reais, sem métricas.") from exc
print("Arquivo oficial presente:", official_zip)

## 2. G1 — Auditoria sem ajustar modelos

Verificamos o alinhamento `X/y/subject`, a estrutura de 561 atributos, a duplicação de nomes de features, as classes 1–6, faltantes, repetição de vetores, pessoas disjuntas entre treino e teste. O arquivo do teste só participa **da inspeção estrutural**, não da escolha de modelos.

In [ ]:
audit_result = audit(ROOT)
assert audit_result["status"] == "PASS"
display(pd.DataFrame([{"corte": k, **{a:v for a,v in item.items() if a in ("shape", "n_subjects", "min_value", "max_value", "nonfinite")}} for k,item in audit_result["splits"].items()]))
print("Nomes de atributos duplicados:", audit_result["duplicated_feature_name_positions"])
print("IDs sobrepostos entre treino/teste:", audit_result["cross_split_subject_overlap"])
print("Duplicatas exatas entre splits:", audit_result["cross_split_exact_duplicate_vectors"])
assert audit_result["cross_split_subject_overlap"] == []

## 3. G2 — EDA do treino, sem antecipar conclusões

A análise por sujeito permite avaliar equilíbrio das seis atividades entre pessoas. Correlações entre colunas **não** justificam eliminar variáveis antes do CV. O teste é ilustrado na contagem de classes apenas como verificação estrutural, não para escolher modelo.

In [ ]:
X_train, y_train, subject_train = load_split(ROOT,"train")
print("X_train", X_train.shape, "pessoas", np.unique(subject_train).size)
ctab = pd.crosstab(pd.Series(subject_train,name="Sujeito"),pd.Series(y_train,name="Atividade"))
ctab.columns=[LABELS[int(k)] for k in ctab.columns]
display(ctab.head(10))
print("Mínimo de janelas por sujeito/atividade:", ctab.min().min())

## 4. G3–G6 — Modelos, validação aninhada e congelamento

A validação **externa** `GroupKFold(5)` gera uma previsão *out-of-fold* para cada janela do treino; a escolha de hiperparâmetros é feita por `GroupKFold(3)` **interno** em cada ajuste. Todos os transformadores aprendem **somente dentro do fold de ajuste**.

**Métrica primária:** $\mathrm{macroF1}=\frac{1}{6}\sum_{c=1}^{6}\frac{2P_cR_c}{P_c+R_c}$ (F1=0 em classe sem predição/suporte). A *balanced accuracy* é a média dos seis recalls.

Famílias: Dummy, regressão logística, k-NN, SVM linear, SVM RBF e Random Forest. Avaliação de H1 usa um modelo fixo independente da seleção de famílias. H2 explora `SelectKBest` somente nos folds; **não** escolhe o vencedor com base no teste.

In [ ]:
lock_file = ROOT / "artifacts/protocol_locked.json"
if lock_file.exists():
    print("Protocolo já congelado; respeitando a seleção gravada (sem retreinar por causa do holdout).")
    lock = json.loads(lock_file.read_text(encoding="utf8"))
else:
    lock = train(ROOT, mode=MODE, run_hypotheses=True)
print("Modelo selecionado (sem usar teste):", lock["selected_model"])
print("Modo originalmente usado:",lock["mode"])
display(pd.read_csv(ROOT/"artifacts/cv_metrics.csv"))

### H1 — Janelas aleatórias versus pessoas não vistas

Contraste em treino oficial: $\Delta_{F1}=F1_{\mathrm{janelas}}-F1_{\mathrm{grupos}}$. Um valor positivo **não identifica causalmente** a sobreposição temporal; pessoa e sessão também contribuem. Um valor nulo ou negativo contraria a hipótese direcional neste protocolo.

In [ ]:
h1_file=ROOT/"artifacts/h1_protocol_contrast.json"
if h1_file.exists():
    h1=json.loads(h1_file.read_text())
    display(pd.DataFrame([h1]).T)
else:
    print("H1 indisponível: protocolo já congelado sem etapa exploratória; não recalcular após lock.")

### H2 — Seleção e estabilidade de atributos

Seleção univariada e escala são ajustadas dentro de cada fold. O índice Jaccard compara posições das 561 features originais (IDs `f001…f561`), mesmo quando seus nomes repetem. A ablação não altera o modelo escolhido para teste final.

In [ ]:
h2_file=ROOT/"artifacts/h2_feature_ablation.csv"
if h2_file.exists():
    display(pd.read_csv(h2_file).query("fold == 'OOF'"))
else:
    print("Sem ablação H2 registrada; não gerar agora após lock.")

## 5. Gráficos de desenvolvimento: F01–F09/F12/F13

São gerados exclusivamente de auditoria e treino. O manifesto registra a proveniência de cada figura. As figuras de teste F10, F11 e F14 só podem existir após a **avaliação final única**.

In [ ]:
images = generate(ROOT)
print("Figuras geradas:", len(images))
for fig in images:
    if not fig["provenance"].startswith("test_"):
        display(Image(filename=str(ROOT/fig["file"]),width=650))

## 6. G7 — Teste oficial único em nove pessoas inéditas

Este bloco somente é autorizado se `protocol_locked.json` existir. Ao iniciar, o programa escreve `test_started.json`; falha após esse marco requer auditoria, **não** retentativa silenciosa. Nunca use os resultados abaixo para trocar o vencedor, grade de parâmetros ou escolha de gráficos.

Bootstrap 95%: **2.000 reamostragens de IDs de sujeito** com reposição (todas as janelas de um sujeito entram juntas, repetidas se ele for sorteado mais de uma vez). Com apenas 9 sujeitos, a faixa é **descritiva e sensível à composição da amostra**.

In [ ]:
test_output = ROOT/"artifacts/test_metrics.json"
if test_output.exists():
    print("Teste pré-existente detectado: resultados carregados SEM reexecutar o holdout.")
    test_result=json.loads(test_output.read_text(encoding="utf8"))
else:
    test_result=final_test(ROOT)
print("Modelo do teste:",test_result["model"],"pessoas:",test_result["n_subjects"])
display(pd.DataFrame(test_result["per_class"]).T)
display(pd.DataFrame([{"macro_f1": test_result["macro_f1"], "balanced_accuracy":test_result["balanced_accuracy"], "media_macro_f1_por_sujeito":test_result["mean_subject_macro_f1"]}]))
print("IC bootstrap por sujeitos:",test_result["bootstrap_subjects"]["global_macro_f1_ci_95"])
print("Erros estático/dinâmico:",test_result["family_errors"])

## 7. G8 — Gráficos finais e discussão científica

A matriz 6×6 exibe suporte por atividade (confusão em contagem e proporcional), taxas por família e incerteza do teste. A síntese interpretativa deve discutir os resultados realmente observados, inclusive hipóteses contrariadas. Este notebook **não escreve uma conclusão favorável antes da medição**.

In [ ]:
all_figures=generate(ROOT)
final_figures=[f for f in all_figures if f["provenance"].startswith("test_")]
for fig in final_figures:
    display(Image(filename=str(ROOT/fig["file"]),width=750))
print("Total de figuras:",len(all_figures))
print("Conclusões dependem de métricas observadas; revisão humana obrigatória para generalização externa.")

## 8. Checklist de anti-leakage e rastreabilidade

- [x] IDs de pessoas do treino e teste disjuntos (verificação automatizada).
- [x] `subject_id` e `row_id` excluídos dos atributos preditores.
- [x] Escala, remoção de variância e seleção ajustadas dentro dos folds.
- [x] `GroupKFold` externo e interno, previsões OOF íntegras.
- [x] Random-window CV apenas em análise H1 no treino oficial.
- [x] Escolha e protocolo com hash de código/dados congelados antes de acessar resultados do teste.
- [x] Teste oficial executado uma única vez; nenhuma otimização pós-teste.
- [x] Intervalos por participante, não por janela.

**Atenção:** a conclusão da execução dependerá dos checks e arquivos realmente gerados. Os checkboxes descrevem garantias implementadas e auditadas pela lógica do repositório, não testes já realizados nos dados UCI neste ambiente de entrega.

**Fontes:** UCI Dataset 240; GroupKFold e Common pitfalls (scikit-learn); Jordao et al. 2019; documentação histórica do dataset. Referências e limitações completas no README e no dossiê.